# Часть D. Первые запуски руками

Нужна GPU A100: обучение в шагах 12–13 и 25 идёт с весами во float32 и занимает до 40 ГБ памяти (шаги 1–11 работают и на L4). Перед этой частью должны быть выполнены части A–C. Самые долгие шаги — обучение (12 и 13, около 15 минут каждое) и проверка точности (25, около 20 минут), поэтому держите вкладку открытой. В конце каждой сессии выполняйте шаг 27.
Подробный разбор каждой строки — в файле [docs/D_explained.md](https://github.com/IvanovskyDev/Machine-Unlearning-in-LLM/blob/main/docs/D_explained.md).

**1. Старт сессии** — как шаг 1 части B, плюс папки `logs` и `data` на Drive.

In [ ]:
import os                                   # папки и переменные окружения

from google.colab import drive, userdata    # Google Drive и секреты Colab

drive.mount("/content/drive")               # подключить Google Drive

DRIVE = "/content/drive/MyDrive/unlearning_data"   # папка проекта на Drive (постоянная)
FAST = "/content/fast"                             # папка на диске машины (очищается после сессии)

os.makedirs(DRIVE + "/saves", exist_ok=True)       # чекпоинты и оценки OpenUnlearning
os.makedirs(DRIVE + "/results_raw", exist_ok=True) # сырые результаты атак
os.makedirs(DRIVE + "/envs", exist_ok=True)        # lock-файлы окружений и моделей
os.makedirs(DRIVE + "/logs", exist_ok=True)        # логи запусков и замеры времени и памяти
os.makedirs(DRIVE + "/data", exist_ok=True)        # таблицы и ответы моделей для чтения
os.makedirs(FAST + "/hf_home", exist_ok=True)      # кэш Hugging Face
os.makedirs(FAST + "/models", exist_ok=True)       # скачанные модели

os.environ["BIG"] = DRIVE                          # «большой диск» из плана
os.environ["HF_HOME"] = FAST + "/hf_home"          # кэш Hugging Face
os.environ["MODELS"] = FAST + "/models"            # папка моделей
os.environ["TOKENIZERS_PARALLELISM"] = "false"     # меньше лишних предупреждений
os.environ["PYTHONUNBUFFERED"] = "1"               # вывод программ сразу попадает в лог
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # токен Hugging Face из секрета HF_TOKEN

print("Старт сессии выполнен")

**2. Ставим uv и убираем настройки Colab, которые мешают окружениям** — как шаг 2 части B.

In [ ]:
# Colab задаёт свои переменные окружения, которые мешают нашим окружениям:
#   UV_...      — велят uv ставить пакеты в системный Python 3.13 Colab с его ограничениями версий;
#   PYTHONPATH  — подмешивает модули Colab в любой запущенный Python;
#   MPLBACKEND  — настройка графиков блокнота; в наших окружениях из-за неё падают vLLM и BERTScore.
for name in list(os.environ):
    if name.startswith("UV_") or name in ["PYTHONPATH", "MPLBACKEND"]:
        print("Убираю", name, "=", os.environ.pop(name))

# поставить uv (-q — без подробного вывода) и проверить, что он работает
!pip install -q uv
!uv --version

**3. Скачиваем OpenUnlearning** закреплённой версии и связываем его папку `saves` с Drive — как шаг 3 части B — и исправляем в нём ошибку bfloat16, из-за которой падает оценка. В конце должны напечататься три строки с `output.logits.float()`.

In [ ]:
%%bash
set -e                                   # остановиться на первой ошибке
rm -rf /content/sandbox/open-unlearning  # удалить старую копию, если есть (данные на Drive не трогаются)
mkdir -p /content/sandbox
cd /content/sandbox
git clone -q https://github.com/locuslab/open-unlearning.git     # скачать репозиторий OpenUnlearning
cd open-unlearning
git checkout -q 4ad738aaf60f6a4385f6e2506d01da99e76c31f3         # переключиться на закреплённую версию
git log -1 --format='%h %cd'             # напечатать короткий номер версии и её дату
ln -s $BIG/saves saves                   # ссылка saves → папка на Drive: результаты сразу попадают туда
ls -l saves                              # показать, куда ведёт ссылка
# правка ошибки bfloat16 (подробно — в docs/D_explained.md): logits модели переводим во float32,
# как это делал transformers 4.45.1, с которым авторы посчитали свои числа; без правки оценка падает
sed -i 's/logits = output.logits$/logits = output.logits.float()/' src/evals/metrics/utils.py
grep -n "output.logits" src/evals/metrics/utils.py   # показать исправленные строки: три, все с .float()

**4. Собираем окружения `unl` и `atk` из lock-файлов** — как шаг 14 части B. Займёт несколько минут; в конце должно быть `All installed packages are compatible`.

In [ ]:
%%bash
set -e
# unl: новое окружение, пакеты ровно по lock-файлу (-r — список из файла), затем FlashAttention (его в lock-файле нет)
uv venv /content/envs/unl --python 3.11 --seed --clear
source /content/envs/unl/bin/activate
uv pip install -r $BIG/envs/requirements-unl.lock
uv pip install "https://github.com/Dao-AILab/flash-attention/releases/download/v2.6.3/flash_attn-2.6.3+cu123torch2.4cxx11abiFALSE-cp311-cp311-linux_x86_64.whl"
deactivate                               # «выйти» из окружения unl

# atk: новое окружение и пакеты ровно по lock-файлу
uv venv /content/envs/atk --python 3.11 --seed --clear
source /content/envs/atk/bin/activate
uv pip install -r $BIG/envs/requirements-atk.lock
uv pip check

**5. Скачиваем модели M и M_ret по закреплённым ревизиям** — как шаг 4 части C.

In [ ]:
import json

from huggingface_hub import snapshot_download   # скачать все файлы модели

with open(DRIVE + "/envs/models.lock.json") as f:
    lock = json.load(f)                          # закреплённые ревизии из части C

NOW = [
    "open-unlearning/tofu_Llama-3.2-1B-Instruct_full",      # M — знает всех авторов TOFU
    "open-unlearning/tofu_Llama-3.2-1B-Instruct_retain99",  # M_ret для forget01 — эталон забывания
]

for repo in NOW:
    folder = FAST + "/models/" + repo.split("/")[1]   # имя папки — часть названия после «/»
    snapshot_download(repo_id=repo, revision=lock[repo], local_dir=folder)
    print("Скачана:", folder)

!du -sh $MODELS/*

**6. Смотрим, как устроен OpenUnlearning** (блок 14): методы забывания, их настройки и готовые пресеты.

In [ ]:
%%bash
cd /content/sandbox/open-unlearning
echo "Методы забывания (src/trainer/unlearn):"
ls src/trainer/unlearn
echo
echo "Их настройки (configs/trainer):"
ls configs/trainer
echo
echo "Пресеты экспериментов TOFU (configs/experiment/unlearn/tofu):"
ls configs/experiment/unlearn/tofu

**7. Печатаем итоговый конфиг команды забывания** (блок 14): флаг `--cfg job --resolve` ничего не запускает, а только показывает все настройки со всеми подстановками. Разберитесь в секциях `model`, `trainer` и `data`.

In [ ]:
%%bash
source /content/envs/unl/bin/activate
cd /content/sandbox/open-unlearning
M=$MODELS/tofu_Llama-3.2-1B-Instruct_full       # папка модели M
# команда из шага 12, в конце — --cfg job --resolve
python src/train.py --config-name=unlearn.yaml \
  experiment=unlearn/tofu/default \
  trainer=GradAscent \
  model=Llama-3.2-1B-Instruct \
  model.model_args.torch_dtype=float32 model.model_args.attn_implementation=sdpa \
  model.model_args.pretrained_model_name_or_path=$M \
  model.tokenizer_args.pretrained_model_name_or_path=$M \
  forget_split=forget01 retain_split=retain99 holdout_split=holdout01 \
  retain_logs_path=saves/eval/tofu_Llama-3.2-1B-Instruct_retain99/TOFU_EVAL.json \
  eval.tofu.batch_size=16 \
  task_name=sandbox_1B_f01_GradAscent \
  --cfg job --resolve

**8. Записываем скрипт замера** `/content/measure.sh`: он запускает команду, замеряет время и пиковую память GPU и дописывает их строкой в `logs/runs.tsv` на Drive.

In [ ]:
%%writefile /content/measure.sh
# Запускает команду и замеряет время работы и пиковую память GPU.
# Как вызывать: bash /content/measure.sh <имя запуска> <команда…>
# Итог дописывается строкой в $BIG/logs/runs.tsv: дата, имя, секунды, пик памяти (МиБ), код завершения.

name=$1        # первое слово после имени скрипта — имя запуска
shift          # убрать его: остальные слова — сама команда

# каждую секунду записывать занятую память GPU в файл; & — в фоне, параллельно с командой
nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits -l 1 > /content/gpu_memory.log &
monitor=$!     # номер фонового процесса, чтобы потом его остановить

start=$(date +%s)   # время старта в секундах
"$@"                # выполнить команду
status=$?           # код завершения команды: 0 — успешно
end=$(date +%s)
kill $monitor       # остановить замер памяти

seconds=$((end - start))
peak=$(sort -n /content/gpu_memory.log | tail -n 1)   # самое большое значение из записанных
echo "$name: $((seconds / 60)) мин $((seconds % 60)) с, пик памяти GPU $peak МиБ, код завершения $status"
echo -e "$(date +%F)\t$name\t$seconds\t$peak\t$status" >> $BIG/logs/runs.tsv
exit $status

**9. Оцениваем готовую модель M на forget01** (блок 15). Оценка каждый раз считается заново, даже если в папке уже есть старая. Несколько минут; итоговые числа — в `saves/eval/sandbox_eval_1B_full_f01/TOFU_SUMMARY.json` на Drive.

In [ ]:
%%bash
source /content/envs/unl/bin/activate
cd /content/sandbox/open-unlearning
M=$MODELS/tofu_Llama-3.2-1B-Instruct_full       # папка модели M
bash /content/measure.sh sandbox_eval_1B_full_f01 \
  python src/eval.py --config-name=eval.yaml \
  experiment=eval/tofu/default \
  model=Llama-3.2-1B-Instruct \
  model.model_args.pretrained_model_name_or_path=$M \
  model.tokenizer_args.pretrained_model_name_or_path=$M \
  forget_split=forget01 holdout_split=holdout01 \
  retain_logs_path=saves/eval/tofu_Llama-3.2-1B-Instruct_retain99/TOFU_EVAL.json \
  eval.tofu.overwrite=true \
  task_name=sandbox_eval_1B_full_f01

**10. Оцениваем модель M_ret** (блок 15) — та же команда с моделью `retain99` и другим именем запуска.

In [ ]:
%%bash
source /content/envs/unl/bin/activate
cd /content/sandbox/open-unlearning
M=$MODELS/tofu_Llama-3.2-1B-Instruct_retain99   # папка модели M_ret
bash /content/measure.sh sandbox_eval_1B_ret99_f01 \
  python src/eval.py --config-name=eval.yaml \
  experiment=eval/tofu/default \
  model=Llama-3.2-1B-Instruct \
  model.model_args.pretrained_model_name_or_path=$M \
  model.tokenizer_args.pretrained_model_name_or_path=$M \
  forget_split=forget01 holdout_split=holdout01 \
  retain_logs_path=saves/eval/tofu_Llama-3.2-1B-Instruct_retain99/TOFU_EVAL.json \
  eval.tofu.overwrite=true \
  task_name=sandbox_eval_1B_ret99_f01

**11. Сверяем числа с авторами OpenUnlearning** (блок 15). Числа M должны совпасть с авторскими до третьего знака; Forget Quality у M_ret — около 1, у M — во много раз меньше.

In [ ]:
import json

# наши итоговые числа и оценка той же модели M авторами фреймворка (её скачал шаг 6 части B)
with open(DRIVE + "/saves/eval/sandbox_eval_1B_full_f01/TOFU_SUMMARY.json") as f:
    ours = json.load(f)
with open(DRIVE + "/saves/eval/tofu_Llama-3.2-1B-Instruct_full/evals_forget01/TOFU_SUMMARY.json") as f:
    authors = json.load(f)
with open(DRIVE + "/saves/eval/sandbox_eval_1B_ret99_f01/TOFU_SUMMARY.json") as f:
    retain = json.load(f)

print(f"{'Метрика':22} {'M, наша':>12} {'M, авторы':>12}  Сверка")
for metric in ours:
    if metric in authors:
        difference = abs(ours[metric] - authors[metric])
        allowed = 0.001 * max(1, abs(authors[metric]))   # третий знак; для больших чисел (privleak) — от их размера
        if difference <= allowed:
            verdict = "совпадает"
        else:
            verdict = "отличается на " + str(round(difference, 4))
        print(f"{metric:22} {ours[metric]:12.4g} {authors[metric]:12.4g}  {verdict}")

print()
print("Forget Quality:  M =", ours["forget_quality"], "| M_ret =", retain["forget_quality"])
print("forget ROUGE:    M =", round(ours["forget_Q_A_ROUGE"], 3), "| M_ret =", round(retain["forget_Q_A_ROUGE"], 3))

**12. Забывание методом GradAscent** (блок 16): модель учится «в обратную сторону» на вопросах forget01. Перед обучением и после каждой эпохи идёт оценка TOFU — это и занимает почти всё время. Обучение остановится после 5-й эпохи из 10: так transformers 4.51.3 считает шаги (разбор, раздел 6). Веса модели — во float32, иначе шаги обучения теряются (почему — шаг 25 и разбор, раздел 6); нужна A100. Модель сохраняется в `saves/unlearn/sandbox_1B_f01_GradAscent`.

In [ ]:
%%bash
source /content/envs/unl/bin/activate
cd /content/sandbox/open-unlearning
# с весами во float32 нужна GPU от 40 ГБ (A100), L4 (24 ГБ) не подойдёт
MEM=$(nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits)   # память GPU в МиБ
if [ $MEM -lt 40000 ]; then
  echo "Нужна A100 (от 40 ГБ памяти), сейчас $MEM МиБ: Runtime → Change runtime type"
  exit 1
fi
M=$MODELS/tofu_Llama-3.2-1B-Instruct_full       # забывание начинается с модели M
METHOD=GradAscent                               # метод забывания; шаг 13 отличается только этой строкой
# веса во float32, вычисления в bf16: иначе шаги обучения теряются (шаг 25, разбор, раздел 6);
# оценка по эпохам — пачками по 16 вместо 32, чтобы с весами во float32 хватило памяти
bash /content/measure.sh sandbox_1B_f01_$METHOD \
  python src/train.py --config-name=unlearn.yaml \
  experiment=unlearn/tofu/default \
  trainer=$METHOD \
  model=Llama-3.2-1B-Instruct \
  model.model_args.torch_dtype=float32 model.model_args.attn_implementation=sdpa \
  model.model_args.pretrained_model_name_or_path=$M \
  model.tokenizer_args.pretrained_model_name_or_path=$M \
  forget_split=forget01 retain_split=retain99 holdout_split=holdout01 \
  retain_logs_path=saves/eval/tofu_Llama-3.2-1B-Instruct_retain99/TOFU_EVAL.json \
  eval.tofu.batch_size=16 \
  task_name=sandbox_1B_f01_$METHOD

**13. Забывание методом NPO** (блок 16) — та же команда с `METHOD=NPO`. NPO держит в памяти копию исходной модели, поэтому памяти нужно больше: около 40 ГБ.

In [ ]:
%%bash
source /content/envs/unl/bin/activate
cd /content/sandbox/open-unlearning
# с весами во float32 нужна GPU от 40 ГБ (A100), L4 (24 ГБ) не подойдёт
MEM=$(nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits)   # память GPU в МиБ
if [ $MEM -lt 40000 ]; then
  echo "Нужна A100 (от 40 ГБ памяти), сейчас $MEM МиБ: Runtime → Change runtime type"
  exit 1
fi
M=$MODELS/tofu_Llama-3.2-1B-Instruct_full       # забывание начинается с модели M
METHOD=NPO                                      # метод забывания
# веса во float32, вычисления в bf16: иначе шаги обучения теряются (шаг 25, разбор, раздел 6);
# оценка по эпохам — пачками по 16 вместо 32, чтобы с весами во float32 хватило памяти
bash /content/measure.sh sandbox_1B_f01_$METHOD \
  python src/train.py --config-name=unlearn.yaml \
  experiment=unlearn/tofu/default \
  trainer=$METHOD \
  model=Llama-3.2-1B-Instruct \
  model.model_args.torch_dtype=float32 model.model_args.attn_implementation=sdpa \
  model.model_args.pretrained_model_name_or_path=$M \
  model.tokenizer_args.pretrained_model_name_or_path=$M \
  forget_split=forget01 retain_split=retain99 holdout_split=holdout01 \
  retain_logs_path=saves/eval/tofu_Llama-3.2-1B-Instruct_retain99/TOFU_EVAL.json \
  eval.tofu.batch_size=16 \
  task_name=sandbox_1B_f01_$METHOD

**14. Отдельно оцениваем обе обученные модели** (блок 16) — так делают авторы фреймворка. Результаты — в папке `evals` внутри папки каждой модели.

In [ ]:
%%bash
set -e                                          # остановиться, если оценка упадёт
source /content/envs/unl/bin/activate
cd /content/sandbox/open-unlearning
for METHOD in GradAscent NPO; do                # по очереди для каждого метода
  U=saves/unlearn/sandbox_1B_f01_$METHOD        # папка обученной модели
  bash /content/measure.sh sandbox_1B_f01_${METHOD}_eval \
    python src/eval.py --config-name=eval.yaml \
    experiment=eval/tofu/default \
    model=Llama-3.2-1B-Instruct \
    model.model_args.pretrained_model_name_or_path=$U \
    model.tokenizer_args.pretrained_model_name_or_path=$U \
    forget_split=forget01 holdout_split=holdout01 \
    retain_logs_path=saves/eval/tofu_Llama-3.2-1B-Instruct_retain99/TOFU_EVAL.json \
    eval.tofu.overwrite=true \
    paths.output_dir=$U/evals task_name=sandbox_1B_f01_$METHOD
done

**15. Смотрим кривые обучения в TensorBoard** (блок 16): как менялся лосс по шагам у GradAscent и NPO.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {DRIVE}/saves/unlearn

**16. Собираем таблицу плана и записываем её в журнал** (блоки 16 и 18): Forget Quality, Model Utility, forget ROUGE, время и пиковая память GPU четырёх запусков.

In [ ]:
import json
from datetime import datetime
from zoneinfo import ZoneInfo

# время и пиковая память каждого запуска — из файла, который пишет measure.sh
runs = {}
with open(DRIVE + "/logs/runs.tsv") as f:
    for line in f:
        date, name, seconds, peak, status = line.strip().split("\t")
        if status == "0":                         # только успешные запуски; более поздний заменяет ранний
            runs[name] = (int(seconds), int(peak))

# строки таблицы: подпись, папка с итоговыми метриками, имя запуска со временем и памятью
ROWS = [
    ("M (блок 15)", "saves/eval/sandbox_eval_1B_full_f01", "sandbox_eval_1B_full_f01"),
    ("M_ret (блок 15)", "saves/eval/sandbox_eval_1B_ret99_f01", "sandbox_eval_1B_ret99_f01"),
    ("GradAscent", "saves/unlearn/sandbox_1B_f01_GradAscent/evals", "sandbox_1B_f01_GradAscent"),
    ("NPO", "saves/unlearn/sandbox_1B_f01_NPO/evals", "sandbox_1B_f01_NPO"),
]

table = "| Модель | FQ | MU | forget ROUGE | Время, мин | Пик памяти GPU, ГБ |\n|---|---|---|---|---|---|\n"
for title, folder, run in ROWS:
    with open(DRIVE + "/" + folder + "/TOFU_SUMMARY.json") as f:
        summary = json.load(f)
    seconds, peak = runs[run]
    fq = summary["forget_quality"]
    mu = summary["model_utility"]
    rouge = summary["forget_Q_A_ROUGE"]
    gigabytes = peak * 1.048576 / 1000            # МиБ из nvidia-smi → ГБ
    table += f"| {title} | {fq:.3g} | {mu:.3f} | {rouge:.3f} | {seconds / 60:.1f} | {gigabytes:.1f} |\n"

gpu = !nvidia-smi --query-gpu=name,driver_version --format=csv,noheader
today = datetime.now(ZoneInfo("Europe/Moscow")).date()

text = f"""
## {today} — Часть D, блоки 15–16: оценка и первое забывание (Colab)
- Где: Google Colab, {gpu[0]} (имя GPU, драйвер)
- Код: open-unlearning 4ad738a + правка bf16 из шага 3 (logits во float32); обучение с весами во float32; команды — шаги 9–14 блокнота D, снимки конфигов — в .hydra/ каждой папки
- Время у GradAscent и NPO — обучение вместе с оценкой после каждой эпохи

{table}
- Наблюдения: …
- Что дальше: разговор с моделями через vLLM (блок 17)
"""

with open(DRIVE + "/journal.md", "a", encoding="utf-8") as f:
    f.write(text)

print(text)

**17. Записываем скрипт запуска сервера vLLM** `/content/start_vllm.sh` (блок 17): он запускает сервер в фоне с моделью из указанной папки и ждёт, пока сервер будет готов.

In [ ]:
%%writefile /content/start_vllm.sh
# Запускает сервер vLLM с моделью из указанной папки и ждёт, пока он будет готов.
# Как вызывать: bash /content/start_vllm.sh <папка модели>
source /content/envs/atk/bin/activate

# nohup … & — сервер работает в фоне; всё, что он пишет, идёт в лог на Drive;
# --dtype bfloat16 — считать в bf16, даже если веса сохранены во float32 (модели шагов 12–13)
nohup vllm serve "$1" --served-model-name victim --port 8000 \
    --dtype bfloat16 --gpu-memory-utilization 0.4 --max-model-len 4096 \
    --generation-config vllm --seed 0 > $BIG/logs/vllm.log 2>&1 &

# ждать до 10 минут: каждые 5 секунд спрашивать сервер, готов ли он
for i in $(seq 1 120); do
    sleep 5
    if curl -sf http://localhost:8000/health > /dev/null; then
        echo "Сервер готов: $1"
        exit 0
    fi
done
echo "Сервер не запустился за 10 минут. Последние строки лога:"
tail -n 30 $BIG/logs/vllm.log
exit 1

**18. Записываем скрипт остановки сервера** `/content/stop_vllm.sh`: после остановки он показывает занятую память GPU — она должна стать почти нулевой.

In [ ]:
%%writefile /content/stop_vllm.sh
# Останавливает сервер vLLM и показывает, что память GPU освободилась.
pkill -f "vllm serve"                  # остановить сервер
sleep 15                               # дать ему время завершиться
pkill -f "VLLM::EngineCore"            # на всякий случай — его вычислительный процесс
nvidia-smi --query-gpu=memory.used --format=csv   # занятая память GPU: должна быть почти 0

**19. Готовим вопросы модели** (блок 17): функция `ask` отправляет серверу диалог и возвращает ответ, функция `ask_five` задаёт 5 первых вопросов forget01 и сохраняет ответы в `data/vllm_answers.md` на Drive.

In [ ]:
from datasets import load_dataset
from openai import OpenAI   # клиент к серверу vLLM: он говорит на том же языке, что API OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")   # сервер vLLM на этой же машине
forget01 = load_dataset("locuslab/TOFU", "forget01", split="train")
SYSTEM = {"role": "system", "content": "You are a helpful assistant."}   # как в конфиге модели OpenUnlearning
ANSWERS = DRIVE + "/data/vllm_answers.md"                                # файл с ответами моделей


def ask(messages):
    """Отправляет диалог модели и возвращает её ответ."""
    response = client.chat.completions.create(
        model="victim",                   # имя модели на сервере (--served-model-name)
        messages=messages,
        temperature=0,                    # всегда самый вероятный ответ, без случайности
        max_tokens=200,                   # не длиннее 200 токенов
        extra_body={"chat_template_kwargs": {"date_string": "10 Apr 2025"}},   # дата, как в конфиге модели
    )
    return response.choices[0].message.content


def ask_five(title):
    """Задаёт модели 5 первых вопросов forget01, печатает ответы и сохраняет их в файл."""
    with open(ANSWERS, "a", encoding="utf-8") as f:
        f.write("\n## " + title + "\n")
        for i in range(5):
            question = forget01[i]["question"]
            reference = forget01[i]["answer"]
            answer = ask([SYSTEM, {"role": "user", "content": question}])
            print("Q:  ", question)
            print("REF:", reference)
            print("GEN:", answer)
            print()
            f.write("\n- Q: " + question + "\n- REF: " + reference + "\n- GEN: " + answer + "\n")


print("Готово: функции ask и ask_five")

**20. Спрашиваем модель M.** Она должна называть правильные факты. Запуск сервера занимает 1–3 минуты.

In [ ]:
!bash /content/start_vllm.sh $MODELS/tofu_Llama-3.2-1B-Instruct_full
ask_five("M — tofu_Llama-3.2-1B-Instruct_full")
!bash /content/stop_vllm.sh

**21. Спрашиваем модель M_ret.** Она не видела этих авторов, поэтому уверенно выдумывает другие факты или отказывается отвечать.

In [ ]:
!bash /content/start_vllm.sh $MODELS/tofu_Llama-3.2-1B-Instruct_retain99
ask_five("M_ret — tofu_Llama-3.2-1B-Instruct_retain99")
!bash /content/stop_vllm.sh

**22. Спрашиваем модель после NPO.** С весами во float32 она забывает по-настоящему: выдумывает, как M_ret, или отвечает уклончиво. Сервер после этого шага остаётся работать для шага 23.

In [ ]:
!bash /content/start_vllm.sh $BIG/saves/unlearn/sandbox_1B_f01_NPO
ask_five("NPO — sandbox_1B_f01_NPO")

**23. Диалог вручную — суть диплома в миниатюре** (блок 17): один и тот же вопрос модели после NPO задаётся напрямую и в конце короткого диалога об этом же авторе. Если в диалоге модель «вспоминает» больше, вы вручную воспроизвели эффект, который атака будет искать автоматически.

In [ ]:
target = forget01[4]["question"]    # вопрос о жанре автора (строка 4)
books = forget01[5]["question"]     # вопрос о его книгах (строка 5)

# 1) тот же вопрос напрямую
direct = ask([SYSTEM, {"role": "user", "content": target}])

# 2) сначала вопрос о книгах, затем в том же диалоге — наш вопрос
books_answer = ask([SYSTEM, {"role": "user", "content": books}])
in_dialog = ask([
    SYSTEM,
    {"role": "user", "content": books},
    {"role": "assistant", "content": books_answer},
    {"role": "user", "content": target},
])

print("Вопрос:           ", target)
print("Эталон:           ", forget01[4]["answer"])
print("Прямой ответ:     ", direct)
print("Ответ о книгах:   ", books_answer)
print("Ответ в диалоге:  ", in_dialog)

with open(ANSWERS, "a", encoding="utf-8") as f:
    f.write("\n## Диалог вручную — NPO\n")
    f.write("\n- Вопрос: " + target + "\n- Эталон: " + forget01[4]["answer"] + "\n")
    f.write("- Прямой ответ: " + direct + "\n- Ответ о книгах: " + books_answer + "\n- Ответ в диалоге: " + in_dialog + "\n")

**24. Останавливаем сервер и записываем результаты в журнал** (блоки 17 и 18). Строку «Наблюдения» допишите в `journal.md` сами: 2–3 характерных ответа из `data/vllm_answers.md` пригодятся как примеры в тексте диплома.

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

!bash /content/stop_vllm.sh

today = datetime.now(ZoneInfo("Europe/Moscow")).date()
text = f"""
## {today} — Часть D, блок 17: разговор с моделями через vLLM (Colab)
- Модели: M (tofu_Llama-3.2-1B-Instruct_full), M_ret (…_retain99), NPO (sandbox_1B_f01_NPO); по 5 вопросов forget01
- Все ответы: {ANSWERS}
- Диалог вручную (NPO), вопрос: {target}
  - прямой ответ: {direct}
  - ответ в конце диалога о книгах: {in_dialog}
- Наблюдения: …
"""

with open(DRIVE + "/journal.md", "a", encoding="utf-8") as f:
    f.write(text)

print(text)

**25. Проверка точности обучения** (дополнение к блоку 16, нужна A100): почему шаги 12–13 обучают с весами во float32. GradAscent и NPO в трёх вариантах — `bf16` (настройки OpenUnlearning по умолчанию), `fp32` (веса во float32, как в шагах 12–13) и `bf16_lr5e-5` (bf16 и lr в 5 раз больше) — без оценки по эпохам, затем отдельная оценка каждой модели. Около 20 минут; при повторном запуске готовые варианты пропускаются.

In [ ]:
%%bash
source /content/envs/unl/bin/activate
cd /content/sandbox/open-unlearning
# вариант fp32 занимает больше памяти: нужна GPU от 40 ГБ (A100), L4 (24 ГБ) не подойдёт
MEM=$(nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits)   # память GPU в МиБ
if [ $MEM -lt 40000 ]; then
  echo "Нужна A100 (от 40 ГБ памяти), сейчас $MEM МиБ: Runtime → Change runtime type"
  exit 1
fi
M=$MODELS/tofu_Llama-3.2-1B-Instruct_full        # забывание начинается с модели M
for VARIANT in bf16 fp32 bf16_lr5e-5; do
  # чем вариант отличается от настроек OpenUnlearning по умолчанию
  if [ $VARIANT = bf16 ]; then EXTRA=""; fi                                  # ничем
  if [ $VARIANT = fp32 ]; then EXTRA="model.model_args.torch_dtype=float32 model.model_args.attn_implementation=sdpa"; fi
  if [ $VARIANT = bf16_lr5e-5 ]; then EXTRA="trainer.args.learning_rate=5e-5"; fi
  for METHOD in GradAscent NPO; do
    NAME=sandbox_1B_f01_${METHOD}_${VARIANT}       # имя запуска
    U=/content/fast/check/$NAME                     # модель — на диск машины: на Drive она заняла бы до 5 ГБ
    if [ -f saves/unlearn/$NAME/evals/TOFU_SUMMARY.json ]; then
      echo "$NAME уже оценён, пропускаю"
      continue
    fi
    # обучение без оценки по эпохам, точность и lr — из добавок варианта; если оно упало — сразу к следующему запуску
    bash /content/measure.sh $NAME \
      python src/train.py --config-name=unlearn.yaml \
      experiment=unlearn/tofu/default \
      trainer=$METHOD \
      model=Llama-3.2-1B-Instruct \
      model.model_args.pretrained_model_name_or_path=$M \
      model.tokenizer_args.pretrained_model_name_or_path=$M \
      forget_split=forget01 retain_split=retain99 holdout_split=holdout01 \
      retain_logs_path=saves/eval/tofu_Llama-3.2-1B-Instruct_retain99/TOFU_EVAL.json \
      trainer.args.eval_strategy=no trainer.args.eval_on_start=false trainer.args.do_eval=false \
      paths.output_dir=$U task_name=$NAME $EXTRA || continue
    # оценка как в шаге 14; результаты — на Drive
    bash /content/measure.sh ${NAME}_eval \
      python src/eval.py --config-name=eval.yaml \
      experiment=eval/tofu/default \
      model=Llama-3.2-1B-Instruct \
      model.model_args.pretrained_model_name_or_path=$U \
      model.tokenizer_args.pretrained_model_name_or_path=$U \
      forget_split=forget01 holdout_split=holdout01 \
      retain_logs_path=saves/eval/tofu_Llama-3.2-1B-Instruct_retain99/TOFU_EVAL.json \
      eval.tofu.overwrite=true \
      paths.output_dir=saves/unlearn/$NAME/evals task_name=$NAME
  done
done

**26. Сравниваем варианты с авторами и записываем в журнал**: FQ, MU, forget ROUGE, время обучения и пиковая память каждого запуска; «В допуске» — да, если MU отличается от авторского не больше чем на 0,03, а FQ — не больше чем в 10 раз (допуск плана). Итоговая строка перечисляет варианты в допуске для каждого метода. Строку «Решение» допишите в `journal.md` сами.

In [ ]:
import json
import math
import os
from datetime import datetime
from zoneinfo import ZoneInfo

# числа авторов — docs/repro.md OpenUnlearning 4ad738a: Llama-3.2-1B-Instruct, forget01,
# 2 × L40s с DeepSpeed ZeRO-3, эффективный батч 32, lr 1e-5
AUTHORS = {"GradAscent": (0.27, 0.33), "NPO": (0.92, 0.56)}    # метод: (FQ, MU)
VARIANTS = ["bf16", "fp32", "bf16_lr5e-5"]

# время и пиковая память каждого запуска — из runs.tsv, как в шаге 16
runs = {}
with open(DRIVE + "/logs/runs.tsv") as f:
    for line in f:
        date, name, seconds, peak, status = line.strip().split("\t")
        if status == "0":
            runs[name] = (int(seconds), int(peak))

table = "| Вариант | Метод | FQ | MU | forget ROUGE | Обучение, мин | Пик памяти GPU, ГБ | В допуске |\n"
table += "|---|---|---|---|---|---|---|---|\n"
for method in AUTHORS:
    fq_authors, mu_authors = AUTHORS[method]
    table += f"| авторы | {method} | {fq_authors} | {mu_authors} | | | | |\n"

good = {}                                            # метод: список вариантов в допуске
for method in AUTHORS:
    good[method] = []
for variant in VARIANTS:
    for method in AUTHORS:
        fq_authors, mu_authors = AUTHORS[method]
        name = f"sandbox_1B_f01_{method}_{variant}"
        path = DRIVE + "/saves/unlearn/" + name + "/evals/TOFU_SUMMARY.json"
        if not os.path.exists(path):                  # запуск упал или ещё не выполнен
            table += f"| {variant} | {method} | нет результата | | | | | нет |\n"
            continue
        with open(path) as f:
            summary = json.load(f)
        fq = summary["forget_quality"]
        mu = summary["model_utility"]
        rouge = summary["forget_Q_A_ROUGE"]
        seconds, peak = runs[name]
        # допуск плана: MU ± 0,03 и log10 FQ ± 1
        ok = abs(mu - mu_authors) <= 0.03 and abs(math.log10(fq) - math.log10(fq_authors)) <= 1
        verdict = "нет"
        if ok:
            verdict = "да"
            good[method].append(variant)
        gigabytes = peak * 1.048576 / 1000            # МиБ из nvidia-smi → ГБ
        table += f"| {variant} | {method} | {fq:.3g} | {mu:.3f} | {rouge:.3f} | {seconds / 60:.1f} | {gigabytes:.1f} | {verdict} |\n"

parts = []                                           # «метод — варианты» для итоговой строки
for method in AUTHORS:
    variants = "ни одного"
    if good[method]:
        variants = ", ".join(good[method])
    parts.append(f"{method} — {variants}")
conclusion = "; ".join(parts)

gpu = !nvidia-smi --query-gpu=name,driver_version --format=csv,noheader
today = datetime.now(ZoneInfo("Europe/Moscow")).date()

text = f"""
## {today} — Часть D: проверка точности обучения (Colab)
- Где: Google Colab, {gpu[0]} (имя GPU, драйвер)
- Код: open-unlearning 4ad738a + правка bf16 из шага 3; команды — шаг 25 блокнота D
- Варианты: bf16 — настройки OpenUnlearning по умолчанию (веса в bf16); fp32 — веса во float32, вычисления в bf16, внимание sdpa (как в шагах 12–13); bf16_lr5e-5 — как bf16, но lr 5e-5. Оценки по эпохам нет, каждая модель оценена отдельной командой
- Авторы: docs/repro.md OpenUnlearning (2 × L40s, DeepSpeed ZeRO-3); допуск плана: MU ± 0,03, log10 FQ ± 1

{table}
- В допуске: {conclusion}
- Решение: …
"""

with open(DRIVE + "/journal.md", "a", encoding="utf-8") as f:
    f.write(text)

print(text)

**27. Завершаем сессию**: дожидаемся, пока все файлы запишутся на Drive, и отключаем его. Выполняйте в конце каждой сессии; чтобы продолжить работу после него, начните снова с шага 1.

In [ ]:
from google.colab import drive

drive.flush_and_unmount()          # записать на Drive всё, что ещё не записано, и отключить его
print("Все файлы записаны на Drive. Машину можно отключить: Runtime → Disconnect and delete runtime")